# Create multi-simulation dataset (Linea 2)
Converts the 4 scaled-hydrograph SFINCS runs (BC x 0.5, 0.75, 1.25, 1.5) onto the
`template_100m.pkl` and merges them into ONE training pkl with 4 events:

    database/datasets/train/ahr_river_v03_marg_additionalsrc_velocity_100m_warmstart_multisim.pkl

The 1.0x event stays OUT of this dataset — it remains the existing
`ahr_river_v03_marg_additionalsrc_velocity_100m_warmstart` pkl, used as validation/test
(config_best_sweep_multisim.yaml with validate_on_test: True).

Per-factor pkls are also kept (train/ and test/) for per-factor evaluation later.
Script twins for hal8: build_template.py (template) + run_convert_multisim.py (conversion+merge).
Same conversion code as create_dataset_100m.ipynb, Step 3.

In [1]:
import os, sys

# Resolve the repo root robustly (works in VS Code and nbconvert, any start cwd)
try:
    REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname(__vsc_ipynb_file__), '..'))
except NameError:
    _here = os.getcwd()
    REPO_ROOT = _here if os.path.isdir(os.path.join(_here, 'database')) \
                else os.path.abspath(os.path.join(_here, '..'))
assert os.path.isdir(os.path.join(REPO_ROOT, 'database')), f'Not the repo root: {REPO_ROOT}'
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print('Repo root:', REPO_ROOT)

Repo root: c:\Users\marrocol\OneDrive - Stichting Deltares\Documents\mSWE-GNN\mSWE-GNN_marg\mSWE-GNN_marg


## Config

In [2]:
TEMPLATE_PKL = 'database/datasets/train/template_100m.pkl'

# Non-warmstart sfincs_map.nc — same grid, stores x/y as 2D arrays (source point coordinates)
SFINCS_MAP_GRID = (
    'database/raw_datasets_ahr/Simulations/'
    'ahr_river_v03_Marg_additionalsrc_velocity_100m_cutpolygon/sfincs_map.nc'
)

SIM_ROOT = 'database/raw_datasets_ahr/Simulations'

# the 4 scaled runs: tag -> simulation folder
SIMS = {
    #'q050': 'ahr_river_v03_Marg_additionalsrc_velocity_100m_cutpolygon_warmstart_q050',
    #'q075': 'ahr_river_v03_Marg_additionalsrc_velocity_100m_cutpolygon_warmstart_q075',
    #'q125': 'ahr_river_v03_Marg_additionalsrc_velocity_100m_cutpolygon_warmstart_q125',
    #'q150': 'ahr_river_v03_Marg_additionalsrc_velocity_100m_cutpolygon_warmstart_q150',
    'q200': 'ahr_river_v03_Marg_additionalsrc_velocity_100m_cutpolygon_warmstart_q200',
    'q300': 'ahr_river_v03_Marg_additionalsrc_velocity_100m_cutpolygon_warmstart_q300',
}

PER_SIM_NAME = 'ahr_river_v03_marg_additionalsrc_velocity_100m_warmstart_{tag}'
MERGED_NAME  = 'ahr_river_v03_marg_additionalsrc_velocity_100m_warmstart_multisim'
# 1.0x dataset, only used here as reference for the Q-peak sanity check
REF_1X_PKL   = 'database/datasets/train/ahr_river_v03_marg_additionalsrc_velocity_100m_warmstart.pkl'

OUT_ROOT = 'database/datasets'

WATER_LEVEL_VAR = 'zs'
BED_LEVEL_VAR   = 'zb'
VX_VAR = 'u'
VY_VAR = 'v'

FORCE_REBUILD_TEMPLATE = False  # careful: rebuilding changes the mesh for ALL datasets
FORCE_REBUILD_PKL      = False  # True to reconvert even if the per-factor pkl exists

for tag, folder in SIMS.items():
    sim_dir = os.path.join(SIM_ROOT, folder)
    ok = all(os.path.exists(os.path.join(sim_dir, f)) for f in ['sfincs_map.nc', 'sfincs.src', 'sfincs.dis'])
    print(f"{tag}: {'OK' if ok else 'MISSING FILES'}  {sim_dir}")

q200: OK  database/raw_datasets_ahr/Simulations\ahr_river_v03_Marg_additionalsrc_velocity_100m_cutpolygon_warmstart_q200
q300: OK  database/raw_datasets_ahr/Simulations\ahr_river_v03_Marg_additionalsrc_velocity_100m_cutpolygon_warmstart_q300


## Step 1 — Create template (if missing)
Same parameters as `build_template.py` / create_dataset_100m.ipynb: SFINCS 100 m grid as finest
level + gmsh meshes at 500/1000/2000 m (4 scales total). Skipped if the template already exists —
the multisim events MUST live on the same template as the 1.0x dataset, so only force-rebuild it
if you are rebuilding ALL datasets.

In [3]:
from database.create_mesh_template_marg import create_mesh_template_pkl

# same sources as build_template.py
SFINCS_DIR_TEMPLATE = ('database/raw_datasets_ahr/Simulations/'
                       'ahr_river_v03_Marg_additionalsrc_velocity_100m_cutpolygon')

if os.path.exists(TEMPLATE_PKL) and not FORCE_REBUILD_TEMPLATE:
    print('Template already exists:', TEMPLATE_PKL)
else:
    create_mesh_template_pkl(
        shapefile_path        = os.path.join(SFINCS_DIR_TEMPLATE, 'gis', 'region.geojson'),
        dem_tif_path          = os.path.join(SFINCS_DIR_TEMPLATE, 'gis', 'dep.tif'),
        output_pkl_path       = TEMPLATE_PKL,
        with_multiscale       = True,
        number_of_multiscales = 4,
        mesh_resolutions      = [2000, 1000, 500],
        sfincs_map_nc         = os.path.join(SFINCS_DIR_TEMPLATE, 'sfincs_map.nc'),
    )
    print('Template created:', TEMPLATE_PKL)

Template already exists: database/datasets/train/template_100m.pkl


## Step 2 — Convert each scaled simulation onto the template
(same code as create_dataset_100m.ipynb Step 3, looped over the 4 runs)

In [4]:
import pickle
import numpy as np
import torch
import xarray as xr

from database.convert_sfincs_to_pkl_marg import (
    load_single_data_object,
    get_target_points,
    get_source_points,
    interpolate_time_series,
    parse_src_file,
    parse_dis_file,
    build_output_data,
)

print('Loading template...')
template_data = load_single_data_object(TEMPLATE_PKL)
target_points = get_target_points(template_data)
print('  Template mesh faces:', target_points.shape[0])

ds_grid = xr.open_dataset(SFINCS_MAP_GRID, decode_times=False)
source_points = get_source_points(ds_grid)
ds_grid.close()
print('  Source points:', source_points.shape[0])

for tag, folder in SIMS.items():
    dataset_name = PER_SIM_NAME.format(tag=tag)
    sim_dir = os.path.join(SIM_ROOT, folder)
    out_train = os.path.join(OUT_ROOT, 'train', dataset_name + '.pkl')
    if not FORCE_REBUILD_PKL and os.path.exists(out_train):
        print('Skipping (already exists):', out_train)
        continue

    print()
    print('Processing:', tag, '->', dataset_name)
    ds = xr.open_dataset(os.path.join(sim_dir, 'sfincs_map.nc'), decode_times=False)

    zs = ds[WATER_LEVEL_VAR].values
    zb = ds[BED_LEVEL_VAR].values
    # SFINCS writes zs=NaN for DRY cells: fill with bed level so they enter as WD=0
    zs_filled = np.where(np.isnan(zs), zb[None, :, :], zs)
    WD_grid = np.maximum(zs_filled - zb[None, :, :], 0.0).astype(np.float32)
    print('  Interpolating WD...')
    WD = interpolate_time_series(source_points, WD_grid, target_points, 'WD')

    ds_raw = xr.open_dataset(os.path.join(sim_dir, 'sfincs_map.nc'), decode_times=False, mask_and_scale=False)
    if VX_VAR and VX_VAR in ds.data_vars:
        VX_raw = ds_raw[VX_VAR].values.astype(np.float32)
        fv = ds_raw[VX_VAR].attrs.get('_FillValue', None)
        if fv is not None:
            VX_raw[VX_raw == fv] = np.nan
        print('  Interpolating VX...')
        VX = interpolate_time_series(source_points, VX_raw, target_points, 'VX')
    else:
        VX = np.zeros_like(WD)
    if VY_VAR and VY_VAR in ds.data_vars:
        VY_raw = ds_raw[VY_VAR].values.astype(np.float32)
        fv = ds_raw[VY_VAR].attrs.get('_FillValue', None)
        if fv is not None:
            VY_raw[VY_raw == fv] = np.nan
        print('  Interpolating VY...')
        VY = interpolate_time_series(source_points, VY_raw, target_points, 'VY')
    else:
        VY = np.zeros_like(WD)
    ds_raw.close()

    time_var = ds.coords.get('time', ds.coords.get('t', None))
    map_times_s = (time_var.values.astype(np.float64) if time_var is not None
                   else np.arange(zs.shape[0]) * 3600.0)
    ds.close()

    print('  Reading src/dis files...')
    src_xy = parse_src_file(os.path.join(sim_dir, 'sfincs.src'))
    dis_times_s, discharge = parse_dis_file(os.path.join(sim_dir, 'sfincs.dis'))
    print(' ', len(src_xy), 'source points, discharge shape:', discharge.shape)

    data_out = build_output_data(
        template_data, WD=WD, VX=VX, VY=VY,
        map_times_s=map_times_s, src_xy=src_xy,
        dis_times_s=dis_times_s, discharge=discharge,
    )

    # save per-factor pkl in BOTH splits (test copy useful for per-factor evaluation later)
    for split in ['train', 'test']:
        os.makedirs(os.path.join(OUT_ROOT, split), exist_ok=True)
        with open(os.path.join(OUT_ROOT, split, dataset_name + '.pkl'), 'wb') as f:
            pickle.dump([data_out], f)
    print('  Saved train+test:', dataset_name + '.pkl')
    print('  WD=', tuple(data_out.WD.shape),
          '| node_BC=', data_out.node_BC.tolist(),
          '| Q peak =', float(data_out.BC[:, :, 1].max()), 'm3/s')

    # sanity: wet-cell count at finest scale
    np_ptr = data_out.node_ptr.numpy()
    wd_s0 = data_out.WD[np_ptr[0]:np_ptr[1]].numpy()
    wet = (wd_s0 > 0.05).sum(0)
    print(f'  wet cells @0.05m: t=0 {wet[0]}  peak {wet.max()} (t={wet.argmax()})  t=-1 {wet[-1]}')

Loading template...
  Template mesh faces: 30079
  Source points: 85656

Processing: q200 -> ahr_river_v03_marg_additionalsrc_velocity_100m_warmstart_q200
  Interpolating WD...
  Interpolating VX...
  Interpolating VY...
  Reading src/dis files...
  7 source points, discharge shape: (481, 7)
  Mapped 7 source points to BOUNDARY mesh nodes: [12555  7287    15   368  5453  3676  6427]
  Saved train+test: ahr_river_v03_marg_additionalsrc_velocity_100m_warmstart_q200.pkl
  WD= (30079, 121) | node_BC= [12555, 7287, 15, 368, 5453, 3676, 6427] | Q peak = 900.0 m3/s
  wet cells @0.05m: t=0 1060  peak 2753 (t=77)  t=-1 2026

Processing: q300 -> ahr_river_v03_marg_additionalsrc_velocity_100m_warmstart_q300
  Interpolating WD...
  Interpolating VX...
  Interpolating VY...
  Reading src/dis files...
  7 source points, discharge shape: (481, 7)
  Mapped 7 source points to BOUNDARY mesh nodes: [12555  7287    15   368  5453  3676  6427]
  Saved train+test: ahr_river_v03_marg_additionalsrc_velocity_1

## Step 3 — Merge the 4 events into one training pkl
Sanity check: the Q peaks must scale ~0.5 / 0.75 / 1.25 / 1.5 relative to the 1.0x run,
and node_BC must be identical for every event (same mesh, same source cells).

In [5]:
# # reference: Q peak of the 1.0x event
# with open(REF_1X_PKL, 'rb') as f:
#     ref_1x = pickle.load(f)
# q_ref = float(ref_1x[0].BC[:, :, 1].max())
# node_bc_ref = ref_1x[0].node_BC.tolist()
# print(f'1.0x reference: Q peak = {q_ref:.1f} m3/s   node_BC = {node_bc_ref}')
# print()

# merged = []
# for tag in SIMS:
#     pkl_path = os.path.join(OUT_ROOT, 'train', PER_SIM_NAME.format(tag=tag) + '.pkl')
#     with open(pkl_path, 'rb') as f:
#         data_list = pickle.load(f)
#     for data in data_list:
#         q_peak = float(data.BC[:, :, 1].max())
#         wd_peak = float(data.WD.max())
#         same_bc = data.node_BC.tolist() == node_bc_ref
#         print(f'{tag}: Q peak = {q_peak:8.1f} m3/s (ratio {q_peak/q_ref:.3f})   '
#               f'WD peak = {wd_peak:.3f} m   node_BC identical: {same_bc}')
#         assert same_bc, f'{tag}: node_BC differs from the 1.0x event!'
#     merged += data_list

# out_path = os.path.join(OUT_ROOT, 'train', MERGED_NAME + '.pkl')
# with open(out_path, 'wb') as f:
#     pickle.dump(merged, f)
# print(f'\nSaved {len(merged)} events -> {out_path}')
# print('Train with config_best_sweep_multisim.yaml (validation/test = the held-out 1.0x event).')